# Impedance ML Engineering — Session 12
## Combine a CNN with metadata and test ablations
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
Does a matrix encoder add information beyond current summaries and known follow-up horizon?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Align matrix and metadata inputs by visit ID.
- Fuse a spatial encoder with a tabular branch.
- Compare controlled feature/modality ablations.
- Interpret multimodal gains without changing evaluation conditions.

## R-to-Python / prerequisite bridge
Multiple predictor blocks → a module taking `(matrix,metadata)`; `torch.cat(...,dim=1)` joins representations, not rows.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Multimodal fusion combines complementary information: a spatial encoder summarizes current matrix shape, while metadata conveys time, horizon, and setting. An ablation tests an information hypothesis by retraining with a controlled input removed. Disabling a branch at inference instead is a distribution-shift experiment.

Useful shape rule: concatenate `(B,72)` and `(B,8)` along feature dimension 1 to get `(B,80)`. Row alignment by visit ID must be verified before tensors discard those IDs. A gradient reaching both branches confirms connectivity; it does not prove both branches add predictive value.

Use `torch.cat([matrix_embedding,metadata_embedding],dim=1)` and a module whose forward accepts both inputs. Share split and budget across ablations.

## Task 1 — Audit alignment (6 min)
Create encoded current metadata using only `age_at_implant_years`, `months_since_implant`, `planned_horizon_months`, `array_type`, `protocol`, `site`. Fit imputation/scaling/encoding on training only. Align rows to matrices by current visit ID; assert unique keys. Predict the metadata width and its relationship to one-hot categories.

## Task 2 — Prepare frozen matrix preprocessing (6 min)
Recreate Session 11 two-channel matrices with training-observed-cell mean/SD. Check matrix, metadata, y, and IDs have identical row order. Test a deliberately shuffled metadata copy and identify an assertion using visit IDs that catches it before IDs are removed.

## Task 3 — Expose a CNN embedding (8 min)
Adapt the CNN to output its 72-dimensional flattened spatial representation before the regression head. Define a metadata branch Linear(metadata_width, 8)→ReLU. Check batch embeddings have shapes (B, 72) and (B, 8). Explain why concatenation should use dim 1.

## Task 4 — Define fusion and count parameters (7 min)
Concatenate embeddings and use Linear(80, 16)→ReLU→Linear(16, 1). Implement forward(matrix,metadata). Predict fused width and total parameter count before executing. Verify gradients reach both branches during one training batch.

## Task 5 — Train the fusion experiment (10 min)
Use TensorDataset(matrix,metadata,y), batch_size 32, Adam(lr=0.01), seed 42, max 60 epochs, patience 8, validation MAE. Adapt the loop unpacking to three tensors. Restore best checkpoint and keep all preprocessing fixed from training. Record time and selected epoch.

## Task 6 — Run targeted ablations (10 min)
Compare full fusion, matrix-only CNN, and metadata-only MLP with the same training/validation groups and budget. Use the current diagonal mean plus horizon ridge as a fourth low-cost baseline. Do not compare different patient splits or report a branch removed only at inference as if it were retrained without that branch.

## Task 7 — Inspect where information helps (5 min)
Build one validation prediction table for all models, calculate MAE by horizon and protocol with patient/row counts. Identify a plausible reason matrix-only forecasting misses a known horizon. Avoid claiming a tiny subgroup contrast is stable.

## Task 8 — Write an ablation decision (0–2 min)
Report modality gains, model complexity, and one alignment risk. State which input set would be required by an inference interface and why a multimodal model that barely improves may be a poor engineering choice.

## Retrieval practice
Explain why target-visit protocol is prohibited but current protocol and planned horizon can be legitimate metadata.

## Professional reasoning
1. What could go wrong when tabular features and matrices are sorted by different keys?
2. Why must a removed branch be retrained for a fair ablation?
3. Could metadata dominate the model even with a powerful matrix encoder?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Add summary features as a third controlled input set and compare full fusion with summaries+metadata. Distinguish representation gain from simply providing more predictors.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)